# Triunfo — fine-tuning para chamadas de ferramentas

Qwen2.5-1.5B-Instruct com adaptadores LoRA e dados preparados do ToolACE. Comparação entre o modelo original e o ajustado, usando os mesmos casos e critérios.

**Sessão atual:** preparação, treinamento e comparação preliminar já concluídos. Execute somente a célula da etapa 8 para continuar. Não use “Executar tudo”: a célula de treinamento impede sobrescrever um resultado existente.

**Persistência:** dados, checkpoints e adaptador ficam em `/content/drive/MyDrive/Triunfo`. Reiniciar o ambiente apaga variáveis da sessão; o modelo salvo deve ser recarregado para avaliar em uma sessão nova.

A inspeção e a preparação original dos dados estão nos notebooks de dados do projeto. Tentativas, diagnósticos e a primeira execução foram preservados em uma cópia de histórico no Drive; não são passos necessários desta sequência.


## 1. Conectar armazenamento

Montar o Drive onde ficam os dados e os resultados.


In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

meu_drive = Path("/content/drive/MyDrive")
assert meu_drive.is_dir(), "O Google Drive não ficou acessível."

PASTA_PERSISTENTE = meu_drive / "Triunfo"
PASTA_PERSISTENTE.mkdir(parents=True, exist_ok=True)

print("Pasta persistente disponível:", PASTA_PERSISTENTE)

## 2. Restaurar dados e verificar ambiente

Conferir os hashes dos três conjuntos; carregar apenas treino e validação. O teste permanece reservado.


In [ ]:
import hashlib
import io
import json
import platform
import tempfile
import zipfile
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError

import torch
from google.colab import files

PASTA_PERSISTENTE = Path("/content/drive/MyDrive/Triunfo")
assert PASTA_PERSISTENTE.is_dir(), "Conecte o Drive primeiro."

# Confirma escrita e leitura no Drive.
with tempfile.NamedTemporaryFile(
    dir=PASTA_PERSISTENTE, suffix=".txt", delete=False
) as arquivo:
    arquivo.write(b"triunfo: verificacao de escrita")
    verificacao = Path(arquivo.name)

assert verificacao.read_bytes() == b"triunfo: verificacao de escrita"
verificacao.unlink()
print("Escrita e leitura no Drive: conferidas")

nome_zip = "triunfo-dados-6bda777-seed42.zip"
zip_drive = PASTA_PERSISTENTE / nome_zip

if zip_drive.is_file():
    conteudo_zip = zip_drive.read_bytes()
else:
    print("\nSelecione o ZIP dos dados preparados:")
    enviados = files.upload()
    assert len(enviados) == 1, "Selecione somente o ZIP."
    conteudo_zip = next(iter(enviados.values()))

hashes_esperados = {
    "train.jsonl": "0a139d3a1b4eb5f0d2c7f854f6e5940570769e19e053f6950290e4ee2314c003",
    "validation.jsonl": "cef828a8d7ca10e0988cc16d056c571e10f13caf6c87458074bae10c2766b954",
    "test.jsonl": "bf4fb9eb1316dfc621eb1b1c1085fbb44d1750b017e5eb65104240bf420f356f",
}

with zipfile.ZipFile(io.BytesIO(conteudo_zip)) as backup:
    restaurados = {
        nome: backup.read(nome)
        for nome in [*hashes_esperados, "manifest.json"]
    }

for nome, esperado in hashes_esperados.items():
    assert hashlib.sha256(restaurados[nome]).hexdigest() == esperado, (
        f"Conteúdo diferente do original: {nome}"
    )
    print(f"{nome}: hash conferido")

manifesto_dados = json.loads(restaurados["manifest.json"])
assert manifesto_dados["dataset_revision"] == (
    "6bda777c88d21e5a204703c1ee45597a8fa4f734"
)

zip_drive.write_bytes(conteudo_zip)
assert hashlib.sha256(zip_drive.read_bytes()).digest() == (
    hashlib.sha256(conteudo_zip).digest()
)
print("\nZIP preservado no Drive:", zip_drive)

PASTA = Path("/content/triunfo_data")
PASTA.mkdir(parents=True, exist_ok=True)
for nome, conteudo in restaurados.items():
    (PASTA / nome).write_bytes(conteudo)

dados_treino = [
    json.loads(linha)
    for linha in restaurados["train.jsonl"].decode("utf-8").splitlines()
]
dados_validacao = [
    json.loads(linha)
    for linha in restaurados["validation.jsonl"].decode("utf-8").splitlines()
]
assert len(dados_treino) == 4469
assert len(dados_validacao) == 559

print("\nTreino:", len(dados_treino))
print("Validação:", len(dados_validacao))
print("Teste: preservado, sem carregar os exemplos")

print("\nAMBIENTE:")
print("Python:", platform.python_version())
for pacote in ["torch", "transformers", "peft", "accelerate", "datasets", "torchao"]:
    try:
        print(f"{pacote}: {version(pacote)}")
    except PackageNotFoundError:
        print(f"{pacote}: não instalado")

print("GPU disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "Memória total:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GiB",
    )

## 3. Carregar modelo e configurar LoRA

Revisão fixa, modelo original congelado, r16/alpha32/dropout0.05 em q/k/v/o. Remover apenas o torchao incompatível. Em uma sessão nova, esta célula inicializa adaptadores novos; não carrega automaticamente os já treinados.


In [ ]:
import sys
import subprocess
import importlib
import json
from pathlib import Path

assert "modelo_lora" not in globals(), (
    "Já existe um modelo LoRA nesta sessão. Não carregue outro."
)

# Repete a correção já verificada na sessão anterior.
subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "torchao"],
    check=True,
)
importlib.invalidate_caches()
assert importlib.util.find_spec("torchao") is None

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed
from peft import (
    LoraConfig, TaskType, get_peft_model,
    get_peft_model_state_dict,
)

assert torch.cuda.is_available(), "A GPU precisa estar ativa."

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
MODEL_REVISION = "989aa7980e4cf806f80c7fef2b1adb7bc71aa306"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    trust_remote_code=False,
    token=False,
)

print("Carregando o Qwen original na GPU do Colab...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    dtype=torch.float16,
    device_map={"": 0},
    attn_implementation="sdpa",
    use_safetensors=True,
    trust_remote_code=False,
    token=False,
)
model.requires_grad_(False)

set_seed(42)
config_lora = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
    init_lora_weights=True,
)

modelo_lora = get_peft_model(model, config_lora)
modelo_lora.eval()

treinaveis = {
    nome: parametro
    for nome, parametro in modelo_lora.named_parameters()
    if parametro.requires_grad
}
assert all("lora_" in nome for nome in treinaveis)
assert sum(p.numel() for p in treinaveis.values()) == 4358144

# Preserva o ponto inicial para conferir antes de iniciar o treino.
estado_inicial = {
    nome: tensor.detach().cpu().clone()
    for nome, tensor in get_peft_model_state_dict(modelo_lora).items()
}

resumo_recuperacao = {
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "rank": 16,
    "alpha": 32,
    "dropout": 0.05,
    "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj"],
    "trainable_parameters": sum(p.numel() for p in treinaveis.values()),
    "gpu": torch.cuda.get_device_name(0),
    "gpu_allocated_gib": round(
        torch.cuda.memory_allocated() / 1024**3, 2
    ),
    "optimizer_steps": 0,
}

destino = Path("/content/drive/MyDrive/Triunfo/recovery_setup.json")
destino.write_text(
    json.dumps(resumo_recuperacao, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print(json.dumps(resumo_recuperacao, ensure_ascii=False, indent=2))
print("Configuração preservada no Drive:", destino)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

{
  "model_id": "Qwen/Qwen2.5-1.5B-Instruct",
  "model_revision": "989aa7980e4cf806f80c7fef2b1adb7bc71aa306",
  "rank": 16,
  "alpha": 32,
  "dropout": 0.05,
  "target_modules": [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj"
  ],
  "trainable_parameters": 4358144,
  "gpu": "Tesla T4",
  "gpu_allocated_gib": 2.89,
  "optimizer_steps": 0
}
Configuração preservada no Drive: /content/drive/MyDrive/Triunfo/recovery_setup.json


## 4. Preparar supervisão

Mascarar o contexto com -100. Treinar somente sobre a resposta e seu encerramento, sem truncar exemplos.


In [ ]:
import json
from pathlib import Path

INSTRUCAO_BASE = (
    "You are a tool-use assistant. Use only the available tools. "
    "When calling a tool, return exactly one call inside square brackets: "
    "[tool_name(argument=value)]. Copy the tool name exactly, including "
    "spaces or dots. Use Python literal syntax for argument values. "
    "If no available tool can fulfill the request, explain why. "
    "If required information is missing, ask for it. "
    "Do not invent missing information or tool results."
)

def montar_mensagens(exemplo, incluir_resposta=False):
    ferramentas = json.dumps(
        exemplo["ferramentas"], ensure_ascii=False, sort_keys=True
    )
    mensagens = [
        {
            "role": "system",
            "content": INSTRUCAO_BASE + "\n\nAvailable tools:\n" + ferramentas,
        },
        {"role": "user", "content": exemplo["pedido"]},
    ]
    if incluir_resposta:
        mensagens.append({
            "role": "assistant",
            "content": exemplo["resposta"],
        })
    return mensagens

exemplos_treino_tokenizados = []

for indice, exemplo in enumerate(dados_treino, start=1):
    contexto = tokenizer.apply_chat_template(
        montar_mensagens(exemplo),
        tokenize=False,
        add_generation_prompt=True,
    )
    completo = tokenizer.apply_chat_template(
        montar_mensagens(exemplo, incluir_resposta=True),
        tokenize=False,
        add_generation_prompt=False,
    )

    contexto_ids = tokenizer(
        contexto, add_special_tokens=False, truncation=False
    )["input_ids"]
    completo_ids = tokenizer(
        completo, add_special_tokens=False, truncation=False
    )["input_ids"]

    n = len(contexto_ids)
    assert completo_ids[:n] == contexto_ids
    assert n < len(completo_ids) <= 2048

    exemplos_treino_tokenizados.append({
        "input_ids": completo_ids,
        "attention_mask": [1] * len(completo_ids),
        "labels": [-100] * n + completo_ids[n:],
    })

    if indice % 1000 == 0:
        print(f"Preparados {indice}/{len(dados_treino)}")

assert len(exemplos_treino_tokenizados) == 4469
assert max(
    len(e["input_ids"]) for e in exemplos_treino_tokenizados
) == 1711

print("Preparação restaurada: 4469 exemplos, sem cortes.")
print("Próxima etapa: treinamento com salvamento no Drive.")


Preparados 1000/4469
Preparados 2000/4469
Preparados 3000/4469
Preparados 4000/4469
Preparação restaurada: 4469 exemplos, sem cortes.
Próxima etapa: treinamento com salvamento no Drive.


## 5. Treinar e preservar o adaptador

Uma época, 559 atualizações, batch físico1/acumulação8, lr1e-4. Checkpoints a cada100 passos e ZIP final diretamente no Drive. O treino abaixo já foi executado; não repetir para continuar a avaliação.


In [ ]:
import gc
import json
import hashlib
import zipfile
import torch
from pathlib import Path
from datasets import Dataset
from transformers import (
    Trainer, TrainingArguments, DataCollatorForSeq2Seq, set_seed,
)
from peft import get_peft_model_state_dict
from google.colab import files

set_seed(42)

raiz_drive = Path("/content/drive/MyDrive/Triunfo")
assert raiz_drive.is_dir(), "Conecte o Google Drive primeiro."

pasta_treino = raiz_drive / "runs" / "lora_epoch1_recovery"
pasta_final = pasta_treino / "adapter_final"

assert not pasta_final.exists(), "Já existe um adaptador final nesta pasta."
assert not list(pasta_treino.glob("checkpoint-*")), (
    "Há checkpoints anteriores. Precisamos retomar o treino."
)
assert len(exemplos_treino_tokenizados) == 4469
assert all(
    torch.equal(t.detach().cpu(), estado_inicial[n])
    for n, t in get_peft_model_state_dict(modelo_lora).items()
), "Os adaptadores não estão no estado inicial."

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

colador = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=None,
    padding=True,
    pad_to_multiple_of=8,
    label_pad_token_id=-100,
    return_tensors="pt",
)

# Confere que o preenchimento preserva a máscara da resposta.
amostra = colador(exemplos_treino_tokenizados[:2])
for i, original in enumerate(exemplos_treino_tokenizados[:2]):
    n = len(original["labels"])
    assert amostra["labels"][i, :n].tolist() == original["labels"]
    assert (amostra["labels"][i, n:] == -100).all().item()
del amostra

argumentos_treino = TrainingArguments(
    output_dir=str(pasta_treino),
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    warmup_steps=28,
    lr_scheduler_type="linear",
    weight_decay=0.01,
    max_grad_norm=1.0,
    fp16=True,
    bf16=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="adamw_torch",
    logging_steps=25,
    logging_nan_inf_filter=False,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    eval_strategy="no",
    report_to="none",
    seed=42,
    data_seed=42,
    dataloader_num_workers=0,
    push_to_hub=False,
)

pasta_treino.mkdir(parents=True, exist_ok=True)
(pasta_treino / "training_arguments.json").write_text(
    argumentos_treino.to_json_string(), encoding="utf-8"
)

modelo_lora.config.use_cache = False
trainer = Trainer(
    model=modelo_lora,
    args=argumentos_treino,
    train_dataset=Dataset.from_list(exemplos_treino_tokenizados),
    data_collator=colador,
    processing_class=tokenizer,
)

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

print("Treino: 4469 exemplos | 1 época | 559 atualizações previstas")
print("Salvamentos no Drive:", pasta_treino)
resultado_treino = trainer.train()

# Salva o resultado final diretamente no Drive.
trainer.save_model(str(pasta_final))
tokenizer.save_pretrained(str(pasta_final))
trainer.save_state()
trainer.save_metrics("train", resultado_treino.metrics)

modelo_lora.gradient_checkpointing_disable()
modelo_lora.config.use_cache = True
modelo_lora.eval()

resumo_treino = {
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "training_examples": 4469,
    "global_step": trainer.state.global_step,
    "epoch": trainer.state.epoch,
    "metrics": resultado_treino.metrics,
    "gpu_peak_allocated_gib": round(
        torch.cuda.max_memory_allocated() / 1024**3, 2
    ),
    "adapter_path": str(pasta_final),
}
(pasta_treino / "training_epoch1.json").write_text(
    json.dumps(resumo_treino, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print(json.dumps(resumo_treino, ensure_ascii=False, indent=2))

# Cria e confere o ZIP, também no Drive.
assert (pasta_final / "adapter_config.json").is_file()
assert (pasta_final / "adapter_model.safetensors").is_file()

arquivos = sorted(
    [p for p in pasta_final.rglob("*") if p.is_file()]
    + list(pasta_treino.glob("*.json"))
)
manifesto_backup = {
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "global_step": trainer.state.global_step,
    "files": {
        p.relative_to(pasta_treino).as_posix():
        hashlib.sha256(p.read_bytes()).hexdigest()
        for p in arquivos
    },
}

zip_final = pasta_treino / "triunfo-lora-epoch1-backup.zip"
with zipfile.ZipFile(zip_final, "w", zipfile.ZIP_DEFLATED) as z:
    for p in arquivos:
        z.write(p, p.relative_to(pasta_treino).as_posix())
    z.writestr(
        "backup_manifest.json",
        json.dumps(manifesto_backup, indent=2),
    )

with zipfile.ZipFile(zip_final) as z:
    for nome, esperado in manifesto_backup["files"].items():
        assert hashlib.sha256(z.read(nome)).hexdigest() == esperado

print("Backup conferido e salvo no Drive:", zip_final)
files.download(str(zip_final))

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 151643}.


Treino: 4469 exemplos | 1 época | 559 atualizações previstas
Salvamentos no Drive: /content/drive/MyDrive/Triunfo/runs/lora_epoch1_recovery


Step,Training Loss
25,0.649668
50,0.486513
75,0.375913
100,0.362670
125,0.405009
150,0.436978
175,0.387154
200,0.372657
225,0.372119
250,0.344457


{
  "model_id": "Qwen/Qwen2.5-1.5B-Instruct",
  "model_revision": "989aa7980e4cf806f80c7fef2b1adb7bc71aa306",
  "training_examples": 4469,
  "global_step": 559,
  "epoch": 1.0,
  "metrics": {
    "train_runtime": 2096.249,
    "train_samples_per_second": 2.132,
    "train_steps_per_second": 0.267,
    "total_flos": 1.9126405725536256e+16,
    "train_loss": 0.374045895762435,
    "epoch": 1.0
  },
  "gpu_peak_allocated_gib": 6.02,
  "adapter_path": "/content/drive/MyDrive/Triunfo/runs/lora_epoch1_recovery/adapter_final"
}
Backup conferido e salvo no Drive: /content/drive/MyDrive/Triunfo/runs/lora_epoch1_recovery/triunfo-lora-epoch1-backup.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Resultado do treinamento

Execução com persistência: 559 atualizações, uma época, tempo 2096,249s, loss média de treino 0,3740459 e pico de memória 6,02GiB. O ZIP final foi conferido no Drive. A loss de treino não demonstra melhoria nas chamadas: a avaliação pós-treino ainda será executada.

## 6. Recuperar a comparação original

Selecione `baseline_validation32.json`, salvo em Downloads. A célula confere o SHA256 e os mesmos 32 exemplos de validação, e preserva o arquivo no Drive. Nenhuma inferência é feita nesta etapa.


In [ ]:
import hashlib
import json
from pathlib import Path
from google.colab import files

pasta_avaliacao = Path('/content/drive/MyDrive/Triunfo/evaluation')
pasta_avaliacao.mkdir(parents=True, exist_ok=True)
arquivo_baseline = pasta_avaliacao / 'baseline_validation32.json'

if not arquivo_baseline.is_file():
    print('Selecione baseline_validation32.json, salvo no seu computador.')
    enviados = files.upload()
    assert len(enviados) == 1, 'Selecione somente o JSON do baseline.'
    conteudo = next(iter(enviados.values()))
else:
    conteudo = arquivo_baseline.read_bytes()

assert hashlib.sha256(conteudo).hexdigest() == (
    'd7c8612328725f87aca42016f025e165f6b6d7f7534699ec3b369c682347fd97'
), 'O arquivo difere do baseline original. Não substituir os resultados.'
baseline_original = json.loads(conteudo)
assert len(baseline_original['results']) == 32
assert len(set(baseline_original['protocol']['example_ids'])) == 32
assert [r['example_id'] for r in baseline_original['results']] == (
    baseline_original['protocol']['example_ids']
)
por_id_validacao = {e['id_entrada']: e for e in dados_validacao}
for registro in baseline_original['results']:
    exemplo = por_id_validacao[registro['example_id']]
    assert registro['reference'] == exemplo['resposta']

arquivo_baseline.write_bytes(conteudo)
assert hashlib.sha256(arquivo_baseline.read_bytes()).hexdigest() == (
    hashlib.sha256(conteudo).hexdigest()
)
print('Baseline original conferido e preservado no Drive.')
print('Casos para comparação: 32; correspondências exatas originais: 16.')
print('Nenhuma geração executada nesta célula.')


Selecione baseline_validation32.json, salvo no seu computador.


Saving baseline_validation32.json to baseline_validation32.json
Baseline original conferido e preservado no Drive.
Casos para comparação: 32; correspondências exatas originais: 16.
Nenhuma geração executada nesta célula.


## 7. Comparar o adaptador com o modelo original

Execute esta célula na sessão que contém o adaptador treinado. Ela confere os pesos contra o arquivo salvo, a revisão do modelo, a instrução e a quantidade de tokens das mesmas32 entradas. Usa o mesmo avaliador e os mesmos parâmetros de geração do baseline.

Não treina nem executa ferramentas externas. Salva cada previsão no Drive e permite retomar os casos concluídos se a execução da célula for interrompida. A sessão precisa conter modelo_lora treinado, tokenizer, montar_mensagens, INSTRUCAO_BASE, dados_validacao e as constantes do modelo.

Esta é uma amostra preliminar pequena de chamadas únicas; resultados não medem a qualidade de respostas textuais ou o desempenho de toda a base.


In [9]:
"""Leitura estrita de chamadas; não executa ferramentas nem valida semântica."""

import ast
import json


def parse_single_call(text, tools):
    """Retorna uma chamada nomeada com valores literais ou None.

    Texto livre, chamadas múltiplas e sintaxes não suportadas retornam None.
    Isso não classifica texto livre como uma recusa correta.
    """
    text = text.strip()
    if not (text.startswith("[") and text.endswith("]")):
        return None
    interior = text[1:-1].strip()
    for tool in sorted(tools, key=lambda item: len(item["name"]), reverse=True):
        name = tool["name"]
        if not interior.startswith(name + "("):
            continue
        try:
            node = ast.parse("function" + interior[len(name):], mode="eval").body
            if not (
                isinstance(node, ast.Call)
                and isinstance(node.func, ast.Name)
                and node.func.id == "function"
            ):
                return None
            if node.args or any(argument.arg is None for argument in node.keywords):
                return None
            arguments = {
                argument.arg: ast.literal_eval(argument.value)
                for argument in node.keywords
            }
            if len(arguments) != len(node.keywords):
                return None
            json.dumps(arguments, allow_nan=False)
            return {"name": name, "arguments": arguments}
        except (SyntaxError, ValueError, TypeError, RecursionError):
            return None
    return None


def score_call(prediction, reference, tools):
    """Compara à referência; igualdade não comprova correção no mundo real.

    Ignora a ordem das chaves. Mantém comparação JSON estrita de valores,
    inclusive distinções entre números, textos e booleanos.
    """
    expected = parse_single_call(reference, tools)
    if expected is None:
        raise ValueError("Referência fora do protocolo de chamada única.")
    observed = parse_single_call(prediction, tools)
    tool_correct = observed is not None and observed["name"] == expected["name"]
    call_exact = tool_correct and json.dumps(
        observed["arguments"], sort_keys=True, ensure_ascii=False, allow_nan=False
    ) == json.dumps(
        expected["arguments"], sort_keys=True, ensure_ascii=False, allow_nan=False
    )
    return {
        "parseable_call": observed is not None,
        "tool_correct": tool_correct,
        "call_exact": call_exact,
    }

import hashlib
import time
from pathlib import Path
from statistics import median

import torch
from peft import get_peft_model_state_dict
from safetensors.torch import load_file
from google.colab import files

pasta_avaliacao = Path('/content/drive/MyDrive/Triunfo/evaluation')
arquivo_baseline = pasta_avaliacao / 'baseline_validation32.json'
conteudo_baseline = arquivo_baseline.read_bytes()
assert hashlib.sha256(conteudo_baseline).hexdigest() == (
    'd7c8612328725f87aca42016f025e165f6b6d7f7534699ec3b369c682347fd97'
)
baseline_original = json.loads(conteudo_baseline)
protocolo_base = baseline_original['protocol']
assert MODEL_ID == protocolo_base['model_id']
assert MODEL_REVISION == protocolo_base['model_revision']
assert hashlib.sha256(INSTRUCAO_BASE.encode('utf-8')).hexdigest() == (
    protocolo_base['instruction_sha256']
)

# Impede avaliar acidentalmente adaptadores novos ou pesos diferentes.
pasta_run = Path('/content/drive/MyDrive/Triunfo/runs/lora_epoch1_recovery')
relatorio_treino = json.loads((pasta_run / 'training_epoch1.json').read_text())
assert relatorio_treino['global_step'] == 559
pesos_path = pasta_run / 'adapter_final' / 'adapter_model.safetensors'
pesos_salvos = load_file(str(pesos_path), device='cpu')
pesos_atuais = get_peft_model_state_dict(modelo_lora)
assert set(pesos_salvos) == set(pesos_atuais)
assert all(
    torch.equal(pesos_atuais[n].detach().cpu(), p)
    for n, p in pesos_salvos.items()
), 'O modelo em memória não corresponde ao adaptador treinado salvo.'
del pesos_salvos, pesos_atuais

por_id_validacao = {e['id_entrada']: e for e in dados_validacao}
baseline_por_id = {r['example_id']: r for r in baseline_original['results']}
ids = protocolo_base['example_ids']
assert len(ids) == len(set(ids)) == 32
parametros = dict(protocolo_base['generation'])
assert parametros['do_sample'] is False
assert parametros['max_new_tokens'] == 768
protocolo = dict(protocolo_base)
protocolo.update({
    'phase': 'lora_epoch1_recovery',
    'adapter_sha256': hashlib.sha256(pesos_path.read_bytes()).hexdigest(),
    'baseline_sha256': hashlib.sha256(conteudo_baseline).hexdigest(),
})
caminho = pasta_avaliacao / 'finetuned_validation32.json'
resultados = []
if caminho.is_file():
    anterior = json.loads(caminho.read_text())
    assert anterior['protocol'] == protocolo, 'Protocolo diferente do salvo.'
    resultados = anterior['results']
concluidos = {r['example_id'] for r in resultados}
assert len(concluidos) == len(resultados)
assert concluidos <= set(ids)

modelo_lora.set_adapter('default')
modelo_lora.eval()
modelo_lora.config.use_cache = True
modelo_lora.gradient_checkpointing_disable()
eos = modelo_lora.generation_config.eos_token_id
ids_fim = eos if isinstance(eos, list) else [eos]
print(f'Casos já concluídos: {len(resultados)}/32')

for identificador in ids:
    if identificador in concluidos:
        continue
    exemplo = por_id_validacao[identificador]
    base = baseline_por_id[identificador]
    assert exemplo['resposta'] == base['reference']
    assert score_call(base['prediction'], base['reference'], exemplo['ferramentas']) == base['metrics']
    texto = tokenizer.apply_chat_template(
        montar_mensagens(exemplo, incluir_resposta=False),
        tokenize=False, add_generation_prompt=True,
    )
    entradas = tokenizer(
        texto, return_tensors='pt', add_special_tokens=False, truncation=False,
    ).to('cuda')
    tamanho_prompt = entradas['input_ids'].shape[1]
    assert tamanho_prompt == base['input_tokens'], 'Tokens de entrada mudaram.'
    torch.cuda.synchronize()
    inicio = time.perf_counter()
    with torch.inference_mode():
        saida = modelo_lora.generate(
            **entradas, **parametros, use_cache=True,
            eos_token_id=eos, pad_token_id=tokenizer.pad_token_id,
        )
    torch.cuda.synchronize()
    segundos = time.perf_counter() - inicio
    gerados = saida[0, tamanho_prompt:].tolist()
    previsao = tokenizer.decode(gerados, skip_special_tokens=True).strip()
    terminou_com_eos = bool(gerados) and gerados[-1] in ids_fim
    resultados.append({
        'example_id': identificador,
        'original_index': exemplo['indice_original'],
        'prediction': previsao,
        'reference': exemplo['resposta'],
        'metrics': score_call(previsao, exemplo['resposta'], exemplo['ferramentas']),
        'input_tokens': tamanho_prompt,
        'generated_tokens': len(gerados),
        'generation_seconds': round(segundos, 3),
        'reached_token_limit': len(gerados) >= 768 and not terminou_com_eos,
    })
    temporario = caminho.with_suffix('.tmp')
    temporario.write_text(json.dumps(
        {'protocol': protocolo, 'results': resultados},
        ensure_ascii=False, indent=2,
    ), encoding='utf-8')
    temporario.replace(caminho)
    print(f'{len(resultados)}/32 concluídos — {segundos:.1f}s', flush=True)

assert len(resultados) == 32
metricas = ['parseable_call', 'tool_correct', 'call_exact']
contagens_base = {
    k: sum(r['metrics'][k] for r in baseline_original['results']) for k in metricas
}
contagens_ajustado = {
    k: sum(r['metrics'][k] for r in resultados) for k in metricas
}
resumo_comparacao = {
    'cases': 32,
    'baseline_counts': contagens_base,
    'finetuned_counts': contagens_ajustado,
    'finetuned_rates': {k: round(v / 32, 4) for k, v in contagens_ajustado.items()},
    'exact_improved_cases': sum(
        r['metrics']['call_exact'] and not baseline_por_id[r['example_id']]['metrics']['call_exact']
        for r in resultados
    ),
    'exact_regressed_cases': sum(
        not r['metrics']['call_exact'] and baseline_por_id[r['example_id']]['metrics']['call_exact']
        for r in resultados
    ),
    'reached_token_limit': sum(r['reached_token_limit'] for r in resultados),
    'median_generation_seconds': round(median(r['generation_seconds'] for r in resultados), 2),
}
(pasta_avaliacao / 'comparison_validation32_summary.json').write_text(
    json.dumps(resumo_comparacao, ensure_ascii=False, indent=2), encoding='utf-8',
)
print('\nCOMPARAÇÃO PRELIMINAR:')
print(json.dumps(resumo_comparacao, ensure_ascii=False, indent=2))
print('Previsões preservadas no Drive:', caminho)
files.download(str(caminho))


Casos já concluídos: 0/32
1/32 concluídos — 1.8s
2/32 concluídos — 2.0s
3/32 concluídos — 0.6s
4/32 concluídos — 0.9s
5/32 concluídos — 1.0s
6/32 concluídos — 0.5s
7/32 concluídos — 1.2s
8/32 concluídos — 0.8s
9/32 concluídos — 1.3s
10/32 concluídos — 2.8s
11/32 concluídos — 1.1s
12/32 concluídos — 1.4s
13/32 concluídos — 1.0s
14/32 concluídos — 4.4s
15/32 concluídos — 0.4s
16/32 concluídos — 1.2s
17/32 concluídos — 0.9s
18/32 concluídos — 1.0s
19/32 concluídos — 1.0s
20/32 concluídos — 0.8s
21/32 concluídos — 1.2s
22/32 concluídos — 0.8s
23/32 concluídos — 1.5s
24/32 concluídos — 0.7s
25/32 concluídos — 5.5s
26/32 concluídos — 3.5s
27/32 concluídos — 0.8s
28/32 concluídos — 1.2s
29/32 concluídos — 2.3s
30/32 concluídos — 0.4s
31/32 concluídos — 0.8s
32/32 concluídos — 8.2s

COMPARAÇÃO PRELIMINAR:
{
  "cases": 32,
  "baseline_counts": {
    "parseable_call": 21,
    "tool_correct": 21,
    "call_exact": 16
  },
  "finetuned_counts": {
    "parseable_call": 30,
    "tool_correct": 30,
 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Resultado preliminar — 32 chamadas

| Métrica | Original | LoRA |
|---|---:|---:|
| Chamada interpretável |21/32|30/32|
| Ferramenta correta na chamada interpretável |21/32|30/32|
| Ferramenta e argumentos iguais à referência |16/32|24/32|

Oito ganhos e zero regressões no critério exato. Isso não exclui piora parcial: o caso 2731 perdeu a estrutura aninhada correta e continuou incorreto nos dois modelos. Um dos oito ganhos (2466) é apenas a distinção estrita entre números inteiros e decimais. Preservar a métrica definida antes do treino e discutir suas limitações.

## 8. Ampliar para todas as chamadas únicas de validação

Comparar os mesmos 430 exemplos entre o modelo original (adaptadores desativados) e o ajustado (adaptadores ativos). Pesos originais permaneceram congelados durante o treinamento. Reaproveitar os 32 resultados reais anteriores e gerar 398 casos novos para cada condição. Não carregar um segundo modelo, treinar novamente nem usar o teste.

Salvar cada previsão no Drive. Se a célula for interrompida, execute a mesma etapa novamente para retomar. Os 129 exemplos textuais de validação ainda precisam de avaliação separada; não interpretar estes resultados como desempenho de todos os formatos.


In [ ]:
import hashlib
import json
import time
import zipfile
from contextlib import nullcontext
from pathlib import Path

import torch
from peft import get_peft_model_state_dict
from safetensors.torch import load_file
from google.colab import files

pasta_eval = Path('/content/drive/MyDrive/Triunfo/evaluation')
base32 = json.loads((pasta_eval / 'baseline_validation32.json').read_text())
lora32 = json.loads((pasta_eval / 'finetuned_validation32.json').read_text())
parametros430 = dict(base32['protocol']['generation'])
assert parametros430 == lora32['protocol']['generation']
assert base32['protocol']['model_id'] == MODEL_ID
assert base32['protocol']['model_revision'] == MODEL_REVISION
assert hashlib.sha256(INSTRUCAO_BASE.encode()).hexdigest() == (
    base32['protocol']['instruction_sha256']
)
pesos_path430 = Path(
    '/content/drive/MyDrive/Triunfo/runs/lora_epoch1_recovery/'
    'adapter_final/adapter_model.safetensors'
)
sha_adapter430 = hashlib.sha256(pesos_path430.read_bytes()).hexdigest()
assert sha_adapter430 == lora32['protocol']['adapter_sha256']
salvos430 = load_file(str(pesos_path430), device='cpu')
atuais430 = get_peft_model_state_dict(modelo_lora)
assert set(salvos430) == set(atuais430)
assert all(torch.equal(atuais430[n].detach().cpu(), p) for n, p in salvos430.items())
assert all(
    not p.requires_grad for n, p in modelo_lora.named_parameters()
    if 'lora_' not in n
), 'Há parâmetros originais treináveis; revisar antes da comparação.'
del salvos430, atuais430

exemplos430 = [e for e in dados_validacao if e['formato'] == 'chamada_unica']
assert len(exemplos430) == 430
ids430 = [e['id_entrada'] for e in exemplos430]
assert len(set(ids430)) == 430
for e in exemplos430:
    assert parse_single_call(e['resposta'], e['ferramentas']) is not None

modelo_lora.set_adapter('default')
modelo_lora.eval()
modelo_lora.config.use_cache = True
modelo_lora.gradient_checkpointing_disable()
eos430 = modelo_lora.generation_config.eos_token_id
fins430 = eos430 if isinstance(eos430, list) else [eos430]
condicoes430 = [('base', base32), ('finetuned', lora32)]
resultados430 = {}

for fase430, preliminar430 in condicoes430:
    protocolo430 = {
        'scope': 'validation_single_call_full',
        'phase': fase430,
        'model_id': MODEL_ID,
        'model_revision': MODEL_REVISION,
        'instruction_sha256': base32['protocol']['instruction_sha256'],
        'generation': parametros430,
        'eos_token_id': eos430,
        'pad_token_id': tokenizer.pad_token_id,
        'example_ids': ids430,
        'reused_preliminary_ids': base32['protocol']['example_ids'],
        'adapter_sha256': sha_adapter430 if fase430 == 'finetuned' else None,
        'base_mode': 'same_frozen_weights_with_adapter_disabled',
    }
    arquivo430 = pasta_eval / f'{fase430}_validation430.json'
    if arquivo430.is_file():
        salvo430 = json.loads(arquivo430.read_text())
        assert salvo430['protocol'] == protocolo430, 'Protocolo mudou; não sobrescrever.'
        lista430 = salvo430['results']
    else:
        # Reaproveita as previsões reais já medidas, sem gerar novamente.
        lista430 = [dict(r, origin='preliminary32') for r in preliminar430['results']]
    por_id430 = {r['example_id']: r for r in lista430}
    assert len(por_id430) == len(lista430)
    assert set(por_id430) <= set(ids430)
    print(f'\n{fase430}: {len(por_id430)}/430 já concluídos', flush=True)

    for exemplo430 in exemplos430:
        id430 = exemplo430['id_entrada']
        if id430 in por_id430:
            registro430 = por_id430[id430]
            assert registro430['reference'] == exemplo430['resposta']
            assert registro430['metrics'] == score_call(
                registro430['prediction'], registro430['reference'], exemplo430['ferramentas']
            )
            continue

        texto430 = tokenizer.apply_chat_template(
            montar_mensagens(exemplo430, incluir_resposta=False),
            tokenize=False, add_generation_prompt=True,
        )
        entrada430 = tokenizer(
            texto430, return_tensors='pt', add_special_tokens=False, truncation=False,
        ).to('cuda')
        n430 = entrada430['input_ids'].shape[1]
        assert n430 + parametros430['max_new_tokens'] <= modelo_lora.config.max_position_embeddings
        torch.cuda.synchronize()
        inicio430 = time.perf_counter()
        contexto430 = modelo_lora.disable_adapter() if fase430 == 'base' else nullcontext()
        with contexto430, torch.inference_mode():
            saida430 = modelo_lora.generate(
                **entrada430, **parametros430, use_cache=True,
                eos_token_id=eos430, pad_token_id=tokenizer.pad_token_id,
            )
        torch.cuda.synchronize()
        segundos430 = time.perf_counter() - inicio430
        gerados430 = saida430[0, n430:].tolist()
        previsao430 = tokenizer.decode(gerados430, skip_special_tokens=True).strip()
        finalizou430 = bool(gerados430) and gerados430[-1] in fins430
        registro430 = {
            'example_id': id430,
            'original_index': exemplo430['indice_original'],
            'prediction': previsao430,
            'reference': exemplo430['resposta'],
            'metrics': score_call(previsao430, exemplo430['resposta'], exemplo430['ferramentas']),
            'input_tokens': n430,
            'generated_tokens': len(gerados430),
            'generation_seconds': round(segundos430, 3),
            'reached_token_limit': len(gerados430) >= parametros430['max_new_tokens'] and not finalizou430,
            'origin': 'full_validation_run',
        }
        lista430.append(registro430)
        por_id430[id430] = registro430
        temporario430 = arquivo430.with_suffix('.tmp')
        temporario430.write_text(json.dumps(
            {'protocol': protocolo430, 'results': lista430},
            ensure_ascii=False, indent=2,
        ), encoding='utf-8')
        temporario430.replace(arquivo430)
        if len(lista430) % 10 == 0 or len(lista430) == 430:
            print(f'{fase430}: {len(lista430)}/430 concluídos', flush=True)

    assert len(por_id430) == 430
    resultados430[fase430] = por_id430

metricas430 = ['parseable_call', 'tool_correct', 'call_exact']
contagens430 = {
    fase: {k: sum(r['metrics'][k] for r in resultados430[fase].values()) for k in metricas430}
    for fase in ['base', 'finetuned']
}
resumo430 = {
    'cases': 430,
    'scope': 'validation_single_call_full',
    'counts': contagens430,
    'rates': {fase: {k: round(v / 430, 4) for k, v in c.items()} for fase, c in contagens430.items()},
    'exact_improved_cases': sum(
        resultados430['finetuned'][i]['metrics']['call_exact']
        and not resultados430['base'][i]['metrics']['call_exact'] for i in ids430
    ),
    'exact_regressed_cases': sum(
        not resultados430['finetuned'][i]['metrics']['call_exact']
        and resultados430['base'][i]['metrics']['call_exact'] for i in ids430
    ),
    'reached_token_limit': {
        fase: sum(r['reached_token_limit'] for r in registros.values())
        for fase, registros in resultados430.items()
    },
    'reused_preliminary_cases_per_condition': 32,
    'textual_validation_examples_not_scored_here': 129,
    'test_used': False,
}
resumo_path430 = pasta_eval / 'comparison_validation430_summary.json'
resumo_path430.write_text(json.dumps(resumo430, ensure_ascii=False, indent=2), encoding='utf-8')
print('\nVALIDAÇÃO — CHAMADAS ÚNICAS:')
print(json.dumps(resumo430, ensure_ascii=False, indent=2))
zip430 = pasta_eval / 'validation430_results.zip'
with zipfile.ZipFile(zip430, 'w', zipfile.ZIP_DEFLATED) as z:
    for nome430 in ['base_validation430.json', 'finetuned_validation430.json', 'comparison_validation430_summary.json']:
        z.write(pasta_eval / nome430, nome430)
with zipfile.ZipFile(zip430) as z:
    assert z.testzip() is None
print('Resultados preservados no Drive:', zip430)
files.download(str(zip430))
